# 06 · Reward hacking: best-of-n against a flawed reward model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/post-training/06_overoptimization.ipynb) &nbsp; 📖 Concept deck: [Reward hacking and overoptimization](https://nilesh-salpe.github.io/visual-ai/post-training/06-reward-hacking/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** best-of-n sampling, its analytic KL, and a simulation where optimizing a proxy reward model first helps and then hurts the true (gold) reward, reproducing Gao et al.'s overoptimization curve.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def log_softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=axis, keepdims=True))

def log_sigmoid(z):
    return -np.logaddexp(0, -z)

def sample_responses(n, rng):
    """Each response has two features: 'helpfulness' h and 'length' L (both standard normal)."""
    return rng.normal(size=(n, 2))

def gold_reward(F):
    """What we actually want: helpfulness counts, extreme length is bad."""
    return F[:, 0] - 0.5 * F[:, 1] ** 2

def proxy_reward(F):
    """A learned reward model that partly mistook length for quality."""
    return F[:, 0] + 1.0 * F[:, 1]

## Task 1: KL of best-of-n

Best-of-n: sample n responses, return the one the reward model scores highest. It is a simple, strong baseline (Llama 2 calls its version *rejection sampling*). How far does it move from the base policy? Analytically (Stiennon et al. 2020, used by Gao et al.):
$$\mathrm{KL}_{bon} = \log n - \frac{n-1}{n}$$

In [ ]:
def kl_bon(n):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_kl_bon(f):
    assert np.isclose(f(1), 0.0)
    assert np.isclose(f(2), np.log(2) - 0.5)
    assert np.isclose(f(1000), np.log(1000) - 0.999)

check("kl_bon", _test_kl_bon, kl_bon)

## Task 2: Pick the best by the proxy

Given proxy scores and gold scores of shape [prompts, n], select per prompt the response with the highest **proxy** score and return the mean proxy and mean **gold** score of the selected responses.

In [ ]:
def best_of_n(proxy, gold):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_best_of_n(f):
    p = np.array([[1, 5, 2], [0, -1, 3]]); g = np.array([[10, 0, 4], [1, 2, -6]])
    assert np.allclose(f(p, g), (4.0, -3.0)), f"got {f(p, g)}"

check("best_of_n", _test_best_of_n, best_of_n)

## Task 3: Sweep n and watch the gold reward

For each n in `ns`, sample `prompts × n` responses, score them with `proxy_reward` and `gold_reward`, run `best_of_n`, and record `(n, kl_bon(n), proxy, gold)`.
Expected (and tested): the proxy rises monotonically, but gold **peaks and then falls**, because large n increasingly selects long responses the proxy over-rewards. That is Goodhart's law: "When a measure becomes a target, it ceases to be a good measure."

In [ ]:
def bon_sweep(ns=(1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024), prompts=2000, seed=0):
    rng = np.random.default_rng(seed); out = []
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_bon_sweep(f):
    res = np.array(f())
    proxy, gold = res[:, 2], res[:, 3]
    assert np.all(np.diff(proxy) > 0), "proxy score must increase with n"
    k = gold.argmax()
    assert 0 < k < len(gold) - 1, f"gold should peak at an intermediate n, peaked at index {k}"
    assert gold[-1] < gold[k] - 0.2, "gold must fall clearly after the peak"

check("bon_sweep", _test_bon_sweep, bon_sweep)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: KL of best-of-n
def kl_bon_solution(n):
    return np.log(n) - (n - 1) / n

check("kl_bon (reference)", _test_kl_bon, kl_bon_solution)

In [ ]:
# @title Solution for Task 2: Pick the best by the proxy
def best_of_n_solution(proxy, gold):
    i = proxy.argmax(1); rows = np.arange(len(proxy))
    return proxy[rows, i].mean(), gold[rows, i].mean()

check("best_of_n (reference)", _test_best_of_n, best_of_n_solution)

In [ ]:
# @title Solution for Task 3: Sweep n and watch the gold reward
def bon_sweep_solution(ns=(1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024), prompts=2000, seed=0):
    rng = np.random.default_rng(seed); out = []
    for n in ns:
        F = sample_responses(prompts * n, rng)
        p, g = proxy_reward(F).reshape(prompts, n), gold_reward(F).reshape(prompts, n)
        mp, mg = best_of_n_solution(p, g)
        out.append((n, kl_bon_solution(n), mp, mg))
    return out

check("bon_sweep (reference)", _test_bon_sweep, bon_sweep_solution)

---
## Experiments
**Fit Gao et al.'s functional form.** They found that for best-of-n the gold score follows $R(d) = d(\alpha - \beta d)$ with $d = \sqrt{\mathrm{KL}}$ (for RL: $d(\alpha - \beta\log d)$). Fit α and β to the simulation by least squares and compare predicted and simulated values. Our toy model isn't theirs, so expect the shape, not their coefficients.

In [ ]:
res = np.array(bon_sweep_solution()); d = np.sqrt(res[:, 1]); gold = res[:, 3] - res[0, 3]
A = np.stack([d, -d ** 2], 1); (a, b), *_ = np.linalg.lstsq(A, gold, rcond=None)
print(f"fit: R(d) = d({a:.3f} - {b:.3f} d)   -> predicted peak at d = {a / (2 * b):.2f} (KL ≈ {(a / (2 * b)) ** 2:.2f} nats)")
for (n, k, p, g), gd in zip(res, gold):
    dd = np.sqrt(k)
    print(f"n={int(n):5d}  KL={k:5.2f}  proxy={p:5.2f}  gold Δ={gd:+.3f}  fit Δ={dd * (a - b * dd):+.3f}")

---
## Takeaways
* Any reward model is a proxy. Optimize it hard enough and you find its mistakes: the proxy score keeps rising while true quality falls.
* KL from the starting policy is the natural "distance optimized". Gao et al. show gold reward vs $\sqrt{\mathrm{KL}}$ has a predictable rise-then-fall shape whose coefficients scale smoothly with reward-model size.
* Mitigations: a KL penalty or early stopping, bigger and better reward models, fresh preference data on the current policy's outputs (Llama 2 and 3 iterate in rounds), and rewards that can be verified (deck 08).